# 03 — Bổ sung tọa độ từ địa chỉ SUUMO

Notebook này giải thích toàn bộ đường đi **địa chỉ → request → JSON → kiểm tra → latitude/longitude → ghép vào listing**. Chạy sau notebook 02. Code geocoding nằm ngay trong các cell, không cần import helper của pipeline.

Dữ liệu hiện tại đến mức **chōme (丁目)**. Các listing cùng chōme nhận cùng điểm; đây không phải tọa độ chính xác từng tòa nhà. Không thêm tên tòa nhà, địa chỉ số nhà hoặc charts.

Provider: [GSI AddressSearch](https://github.com/gsi-cyberjapan/gsimaps/issues/29). Kết quả được cache local; notebook không ghi warehouse và không ghi đè CSV nguồn.

## Bước 1 — Đọc dữ liệu current
`prefecture`, `municipality`, `locality` và `chome` là các thành phần bắt buộc để geocode ở mức chōme. `district` và `ward` có thể trống tùy cách tổ chức hành chính. Chỉ đọc thông tin địa lý công khai phục vụ kiểm tra; không hiển thị phone/remarks.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebook' else Path.cwd() / 'notebook'
DATA_DIR = NOTEBOOK_DIR / 'data'
INPUT_PATH = DATA_DIR / 'suumo_rentals_current.csv'
CACHE_PATH = DATA_DIR / 'suumo_address_geocodes_cache.json'
OUTPUT_PATH = DATA_DIR / 'suumo_rentals_current_geocoded.csv'
ADDRESS_OUTPUT_PATH = DATA_DIR / 'suumo_address_geocodes.csv'
REPORT_PATH = DATA_DIR / 'suumo_geocode_report.json'

current = pd.read_csv(INPUT_PATH, encoding='utf-8-sig', low_memory=False,
                      dtype={'suumo_property_code': 'string', 'task_id': 'string'})
print(f'Input: {INPUT_PATH.resolve()} | {len(current):,} listings')
display(current[['prefecture', 'district', 'municipality', 'ward', 'locality', 'chome']].head())

Input: /Users/hoangvinh/Library/CloudStorage/OneDrive-Personal/Workspace/japan_analysis/notebook/data/suumo_rentals_current.csv | 17,380 listings


,prefecture,district,municipality,ward,locality,chome
0,大阪府,NaN,大阪市,淀川区,東三国,2
1,大阪府,NaN,大阪市,淀川区,西三国,3
2,大阪府,NaN,大阪市,淀川区,西三国,3
3,大阪府,NaN,大阪市,淀川区,三津屋中,1
4,大阪府,NaN,大阪市,淀川区,加島,1


## Bước 2 — Hàm chuẩn hóa và geocoder
- NFKC chuyển `１` thành `1`; các cột đã parse được ghép thành `大阪府大阪市淀川区西宮原1丁目`.
- Không suy diễn phần bị thiếu. Chỉ chuẩn hóa số trong **hậu tố chōme**, giữ nguyên tên địa danh như `東三国`.
- GSI trả GeoJSON với `coordinates = [longitude, latitude]`: kinh độ trước, vĩ độ sau.
- Chỉ nhận địa chỉ khớp, geometry Point và điểm hợp lệ ở Nhật. Không chọn mặc định kết quả đầu tiên hoặc điểm mức thành phố/quận.
- Request chạy tuần tự, cách tối thiểu 1 giây, timeout 15 giây; tối đa 3 retry cho lỗi tạm thời, có backoff/Retry-After.
- `matched` được tái sử dụng; lỗi request thử lại sau 1 giờ, không tìm thấy sau 7 ngày. Kết quả mơ hồ/không hợp lệ chờ sửa rule hoặc `force=True`.

In [2]:
"""GSI AddressSearch: chuẩn hóa chōme và chỉ nhận kết quả cùng địa chỉ."""
import json
import math
import re
import time
import unicodedata
from datetime import datetime, timedelta, timezone
from email.utils import parsedate_to_datetime
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

PROVIDER = 'gsi_address_search'
QUERY_VERSION = 'chome_v1'
ENDPOINT = 'https://msearch.gsi.go.jp/address-search/AddressSearch'
ADDRESS_FIELDS = ('prefecture', 'district', 'municipality', 'ward', 'locality', 'chome')


def clean_part(value):
    """NFKC rồi bỏ khoảng trắng; không sửa chữ số nằm trong tên địa danh."""
    if value is None:
        return ''
    text = unicodedata.normalize('NFKC', str(value))
    text = re.sub(r'\s+', '', text)
    return '' if text in {'', 'nan', 'NaN', '<NA>', 'None', '-', '―'} else text


def build_address_query(parts):
    """Thiếu prefecture/municipality/locality/chōme thì không suy diễn địa chỉ."""
    values = {field: clean_part(parts.get(field)) for field in ADDRESS_FIELDS}
    chome = values['chome']
    if not all(values[field] for field in ('prefecture', 'municipality', 'locality')):
        return None
    if not re.fullmatch(r'[0-9]+(?:\.0+)?', chome):
        return None
    number = int(chome.split('.')[0])
    if not 1 <= number <= 9999:
        return None
    return ''.join(values[field] for field in ADDRESS_FIELDS[:-1]) + str(number) + '丁目'


def kanji_integer(text):
    digits = {'〇': 0, '零': 0, '一': 1, '二': 2, '三': 3, '四': 4,
              '五': 5, '六': 6, '七': 7, '八': 8, '九': 9}
    units = {'十': 10, '百': 100, '千': 1000}
    if text.isascii() and text.isdigit():
        return int(text)
    if all(char in digits for char in text):
        return int(''.join(str(digits[char]) for char in text))
    total, current = 0, 0
    for char in text:
        if char in digits:
            current = digits[char]
        elif char in units:
            total += (current or 1) * units[char]
            current = 0
        else:
            raise ValueError('Invalid chome numeral')
    return total + current


def canonical_address(text):
    """Chỉ đổi số ngay trước 丁目 ở cuối chuỗi, ví dụ giữ nguyên 東三国."""
    text = clean_part(text)
    match = re.search(r'([0-9〇零一二三四五六七八九十百千]+)丁目$', text)
    if match:
        text = text[:match.start()] + str(kanji_integer(match.group(1))) + '丁目'
    return text


def result_record(address_query, status, **extra):
    return {'provider': PROVIDER, 'query_version': QUERY_VERSION,
            'address_query': address_query, 'status': status,
            'latitude': None, 'longitude': None, 'matched_address': None,
            'coordinate_level': None, 'raw_response': None, 'error_message': None,
            'geocoded_at': None, **extra}


def select_geocode_result(address_query, payload):
    """Không lấy kết quả đầu tiên; cần địa chỉ khớp và Point hợp lệ tại Nhật."""
    if not isinstance(payload, list):
        return result_record(address_query, 'invalid_response', raw_response=payload,
                             error_message='Expected a GeoJSON feature list')
    matches, invalid = {}, False
    for feature in payload:
        if not isinstance(feature, dict):
            invalid = True
            continue
        properties = feature.get('properties')
        geometry = feature.get('geometry')
        if not isinstance(properties, dict) or not isinstance(geometry, dict):
            invalid = True
            continue
        title = properties.get('title')
        if not isinstance(title, str) or canonical_address(title) != canonical_address(address_query):
            continue
        coordinates = geometry.get('coordinates')
        if (geometry.get('type') != 'Point' or not isinstance(coordinates, list)
                or len(coordinates) != 2 or any(isinstance(n, bool) or not isinstance(n, (int, float))
                                               or not math.isfinite(n) for n in coordinates)):
            invalid = True
            continue
        longitude, latitude = coordinates  # GeoJSON trả [kinh độ, vĩ độ].
        # Giới hạn toàn cầu + sanity check Nhật để bắt đảo thứ tự/điểm ở nước khác.
        if not (-180 <= longitude <= 180 and -90 <= latitude <= 90
                and 122 <= longitude <= 154 and 20 <= latitude <= 46):
            invalid = True
            continue
        matches[(float(longitude), float(latitude))] = title
    if invalid:
        return result_record(address_query, 'invalid_response', raw_response=payload,
                             error_message='Malformed candidate or invalid coordinates')
    if len(matches) > 1:
        return result_record(address_query, 'ambiguous', raw_response=payload,
                             error_message='Multiple matching addresses have different points')
    if not matches:
        return result_record(address_query, 'not_found', raw_response=payload,
                             error_message='No exact chome address match')
    (longitude, latitude), title = next(iter(matches.items()))
    return result_record(address_query, 'matched', raw_response=payload,
                         latitude=latitude, longitude=longitude, matched_address=title,
                         coordinate_level='chome', geocoded_at=datetime.now(timezone.utc).isoformat())


def reject_nonfinite_json(value):
    raise ValueError('Non-finite JSON number: ' + value)


def finite_json_float(value):
    number = float(value)
    if not math.isfinite(number):
        raise ValueError('Non-finite JSON float: ' + value)
    return number


class GSIClient:
    """Rate limit áp dụng cho mọi request, kể cả retry."""
    def __init__(self, *, timeout=15, retries=3, min_interval=1.0,
                 opener=urlopen, sleep=time.sleep, monotonic=time.monotonic):
        if timeout <= 0 or retries < 0 or min_interval < 1:
            raise ValueError('timeout > 0, retries >= 0, min_interval >= 1 required')
        self.timeout, self.retries, self.min_interval = timeout, retries, min_interval
        self.opener, self.sleep, self.monotonic = opener, sleep, monotonic
        self.last_request = None

    def request_json(self, address_query):
        url = ENDPOINT + '?' + urlencode({'q': address_query})
        for attempt in range(self.retries + 1):
            if self.last_request is not None:
                self.sleep(max(0, self.min_interval - (self.monotonic() - self.last_request)))
            self.last_request = self.monotonic()
            try:
                request = Request(url, headers={'User-Agent': 'japan-analysis/1.0 (address enrichment)',
                                               'Accept': 'application/json'})
                with self.opener(request, timeout=self.timeout) as response:
                    return json.load(response, parse_constant=reject_nonfinite_json, parse_float=finite_json_float), attempt + 1
            except HTTPError as exc:
                if exc.code not in {429, 500, 502, 503, 504} or attempt == self.retries:
                    raise
                delay = 2 ** attempt
                retry_after = exc.headers.get('Retry-After') if exc.headers else None
                if retry_after:
                    try:
                        delay = max(delay, float(retry_after))
                    except ValueError:
                        try:
                            delay = max(delay, (parsedate_to_datetime(retry_after)
                                                - datetime.now(timezone.utc)).total_seconds())
                        except (ValueError, TypeError):
                            pass
                self.sleep(delay)
            except (URLError, TimeoutError, OSError):
                if attempt == self.retries:
                    raise
                self.sleep(2 ** attempt)


def geocode_address(address_query, client=None):
    if not address_query:
        return result_record(address_query, 'invalid_input', error_message='Missing chome address')
    client = client or GSIClient()
    try:
        payload, requests = client.request_json(address_query)
    except (HTTPError, URLError, TimeoutError, OSError) as exc:
        return result_record(address_query, 'request_error', error_message=str(exc)[:1000])
    except (ValueError, UnicodeError) as exc:
        return result_record(address_query, 'invalid_response', error_message=str(exc)[:1000])
    return {**select_geocode_result(address_query, payload), 'request_count': requests}


def cache_is_due(cached, now=None, force=False):
    if force or not cached:
        return True
    if cached['status'] == 'matched':
        return False
    due = cached.get('next_retry_at')
    if not due:
        return False
    if isinstance(due, str):
        due = datetime.fromisoformat(due)
    return due <= (now or datetime.now(timezone.utc))


def merge_cached_result(previous, result, now=None):
    """Force thất bại giữ nguyên điểm đã được nhận; lưu lỗi của lần thử mới."""
    now = now or datetime.now(timezone.utc)
    result = {**result, 'attempt_count': (previous or {}).get('attempt_count', 0) + 1,
              'last_attempted_at': now.isoformat(), 'next_retry_at': None}
    if previous and previous['status'] == 'matched' and result['status'] != 'matched':
        return {**previous, 'attempt_count': result['attempt_count'],
                'last_attempted_at': result['last_attempted_at'],
                'error_message': 'Refresh ' + result['status'] + ': ' + (result.get('error_message') or ''),
                'last_attempt_status': result['status'], 'last_attempt_response': result.get('raw_response')}
    result['last_attempt_status'] = result['status']
    result['last_attempt_response'] = result.get('raw_response')
    days = 7 if result['status'] == 'not_found' else None
    if days:
        result['next_retry_at'] = (now + timedelta(days=days)).isoformat()
    elif result['status'] == 'request_error':
        result['next_retry_at'] = (now + timedelta(hours=1)).isoformat()
    return result


## Bước 3 — Tạo danh sách địa chỉ duy nhất
Một địa chỉ được hỏi một lần rồi ghép cho nhiều listing. Không dùng mã listing làm khóa cache.

In [3]:
working = current.copy()
working['geocode_address_query'] = working.apply(lambda row: build_address_query(row), axis=1)
queries = sorted(working['geocode_address_query'].dropna().unique())
print(f'{len(queries):,} địa chỉ duy nhất; {working.geocode_address_query.isna().sum():,} listing thiếu đầu vào')
display(working[['address_raw', 'geocode_address_query']].drop_duplicates().head(10))
assert build_address_query(dict(prefecture='大阪府', municipality='大阪市', ward='淀川区',
                               locality='西宮原', chome='１')) == '大阪府大阪市淀川区西宮原1丁目'
assert canonical_address('大阪府大阪市淀川区東三国二丁目') == '大阪府大阪市淀川区東三国2丁目'
assert build_address_query(dict(prefecture='大阪府', municipality='大阪市', locality='東三国')) is None

87 địa chỉ duy nhất; 0 listing thiếu đầu vào


,address_raw,geocode_address_query
0,大阪府大阪市淀川区東三国２,大阪府大阪市淀川区東三国2丁目
1,大阪府大阪市淀川区西三国３,大阪府大阪市淀川区西三国3丁目
3,大阪府大阪市淀川区三津屋中１,大阪府大阪市淀川区三津屋中1丁目
4,大阪府大阪市淀川区加島１,大阪府大阪市淀川区加島1丁目
5,大阪府大阪市淀川区加島４,大阪府大阪市淀川区加島4丁目
14,大阪府大阪市淀川区十三元今里１,大阪府大阪市淀川区十三元今里1丁目
16,大阪府大阪市淀川区三国本町２,大阪府大阪市淀川区三国本町2丁目
18,大阪府大阪市淀川区宮原２,大阪府大阪市淀川区宮原2丁目
21,大阪府大阪市淀川区宮原１,大阪府大阪市淀川区宮原1丁目
23,大阪府大阪市淀川区西三国４,大阪府大阪市淀川区西三国4丁目


## Bước 4 — Cache local và lưu sau mỗi địa chỉ
Cache phân biệt provider + phiên bản rule + địa chỉ. Việc ghi dùng file tạm rồi thay nguyên tử, nên có thể chạy tiếp sau khi bị ngắt. Muốn truy vấn lại một điểm đã thành công, gọi `resolve_query(query, force=True)`; nếu refresh lỗi, giữ lại điểm cũ và lưu lỗi mới.

`cache_hits` đếm lượt truy cập cache, gồm cả địa chỉ ví dụ và 5 mẫu; vì vậy có thể lớn hơn số địa chỉ duy nhất. `lookups` đếm lượt geocode thực hiện trong lần chạy notebook.

In [4]:
cache = json.loads(CACHE_PATH.read_text(encoding='utf-8')) if CACHE_PATH.exists() else {}
client = GSIClient()
run_counts = {'cache_hits': 0, 'lookups': 0}

def cache_key(query):
    return json.dumps([PROVIDER, QUERY_VERSION, query], ensure_ascii=False)

def save_cache():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    temporary = CACHE_PATH.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(CACHE_PATH)

def resolve_query(query, force=False):
    key = cache_key(query)
    previous = cache.get(key)
    if not cache_is_due(previous, force=force):
        run_counts['cache_hits'] += 1
        return previous
    result = merge_cached_result(previous, geocode_address(query, client))
    cache[key] = result
    save_cache()
    run_counts['lookups'] += 1
    return result

## Bước 5 — Xem chi tiết một request
Ví dụ lần thử trước: `大阪府大阪市淀川区西宮原1丁目` → latitude `34.732430`, longitude `135.492401`. Cell dưới lưu kết quả thực tế của lần chạy này, có thể dùng cache khi đã chạy trước.

Các trường `matched_address`, `coordinate_level` và `status` giúp phân biệt có tọa độ với có điểm đúng địa chỉ yêu cầu.

In [5]:
example_query = '大阪府大阪市淀川区西宮原1丁目'
example = resolve_query(example_query)
print('Địa chỉ:', example_query)
print('URL:', ENDPOINT + '?' + urlencode({'q': example_query}))
print('JSON phản hồi:')
print(json.dumps(example['raw_response'], ensure_ascii=False, indent=2))
display(pd.DataFrame([{k: example.get(k) for k in
                      ['matched_address', 'latitude', 'longitude', 'coordinate_level', 'status']}]))

Địa chỉ: 大阪府大阪市淀川区西宮原1丁目
URL: https://msearch.gsi.go.jp/address-search/AddressSearch?q=%E5%A4%A7%E9%98%AA%E5%BA%9C%E5%A4%A7%E9%98%AA%E5%B8%82%E6%B7%80%E5%B7%9D%E5%8C%BA%E8%A5%BF%E5%AE%AE%E5%8E%9F1%E4%B8%81%E7%9B%AE
JSON phản hồi:
[
  {
    "geometry": {
      "coordinates": [
        135.492401,
        34.73243
      ],
      "type": "Point"
    },
    "type": "Feature",
    "properties": {
      "addressCode": "",
      "title": "大阪府大阪市淀川区西宮原一丁目"
    }
  }
]


,matched_address,latitude,longitude,coordinate_level,status
0,大阪府大阪市淀川区西宮原一丁目,34.73243,135.492401,chome,matched


## Bước 6 — Chạy 5 địa chỉ mẫu rồi toàn bộ địa chỉ duy nhất
Hai cell tách riêng để bạn có thể đọc kết quả mẫu trước. Cache được lưu ngay sau từng địa chỉ; chạy lại không gọi lại các địa chỉ thành công.

In [6]:
sample_results = [resolve_query(query) for query in queries[:5]]
display(pd.DataFrame(sample_results)[['address_query', 'matched_address', 'latitude', 'longitude', 'status']])

,address_query,matched_address,latitude,longitude,status
0,大阪府大阪市淀川区三国本町1丁目,大阪府大阪市淀川区三国本町一丁目,34.731243,135.487366,matched
1,大阪府大阪市淀川区三国本町2丁目,大阪府大阪市淀川区三国本町二丁目,34.734680,135.486267,matched
2,大阪府大阪市淀川区三国本町3丁目,大阪府大阪市淀川区三国本町三丁目,34.737350,135.486221,matched
3,大阪府大阪市淀川区三津屋中1丁目,大阪府大阪市淀川区三津屋中一丁目,34.728222,135.472641,matched
4,大阪府大阪市淀川区三津屋中2丁目,大阪府大阪市淀川区三津屋中二丁目,34.727283,135.468674,matched


In [7]:
results = []
for index, query in enumerate(queries, start=1):
    result = resolve_query(query)
    results.append(result)
    if index % 10 == 0 or index == len(queries):
        print(f'{index}/{len(queries)} địa chỉ | cache hits={run_counts["cache_hits"]} | lookups={run_counts["lookups"]}')
address_results = pd.DataFrame(results, columns=['provider', 'query_version', 'address_query', 'status',
    'latitude', 'longitude', 'matched_address', 'coordinate_level', 'geocoded_at', 'error_message'])
display(address_results.groupby('status', dropna=False).size().rename('addresses').to_frame())

10/87 địa chỉ | cache hits=16 | lookups=0
20/87 địa chỉ | cache hits=26 | lookups=0
30/87 địa chỉ | cache hits=36 | lookups=0
40/87 địa chỉ | cache hits=46 | lookups=0
50/87 địa chỉ | cache hits=56 | lookups=0
60/87 địa chỉ | cache hits=66 | lookups=0
70/87 địa chỉ | cache hits=76 | lookups=0
80/87 địa chỉ | cache hits=86 | lookups=0
87/87 địa chỉ | cache hits=93 | lookups=0


,addresses
status,
matched,87


## Bước 7 — Ghép tọa độ, sanity checks và coverage
Merge `many_to_one` bắt lỗi nếu bảng tọa độ có nhiều dòng cho một địa chỉ. Listing thiếu địa chỉ hoặc không geocode được vẫn được giữ lại, tọa độ NULL. Các khóa và thứ tự dòng nguồn phải giữ nguyên.

In [8]:
geo_columns = address_results.rename(columns={'address_query': 'geocode_address_query',
    'status': 'geocode_status', 'provider': 'geocode_provider', 'query_version': 'geocode_query_version',
    'matched_address': 'geocode_matched_address', 'error_message': 'geocode_error_message'})
enriched = working.merge(geo_columns, on='geocode_address_query', how='left', validate='many_to_one', sort=False)
enriched.loc[enriched.geocode_address_query.isna(), 'geocode_status'] = 'invalid_input'
assert len(enriched) == len(current)
for key in ['task_id', 'suumo_property_code']:
    if key in current:
        pd.testing.assert_series_equal(current[key], enriched[key], check_names=True)
assert enriched.latitude.notna().equals(enriched.longitude.notna())
matched = enriched.geocode_status.eq('matched')
assert enriched.loc[matched, 'latitude'].between(20, 46).all()
assert enriched.loc[matched, 'longitude'].between(122, 154).all()
assert enriched.loc[matched, 'coordinate_level'].eq('chome').all()
assert enriched.loc[~matched, ['latitude', 'longitude']].isna().all().all()
report = {'provider': PROVIDER, 'query_version': QUERY_VERSION, 'generated_at': datetime.now(timezone.utc).isoformat(),
    'listing_count': len(enriched), 'unique_address_count': len(queries),
    'matched_address_count': int(address_results.status.eq('matched').sum()),
    'matched_listing_count': int(matched.sum()), 'invalid_input_listing_count': int(enriched.geocode_address_query.isna().sum()),
    'address_coverage': float(address_results.status.eq('matched').mean()) if len(queries) else 0.0,
    'listing_coverage': float(matched.mean()) if len(enriched) else 0.0,
    'address_status_counts': address_results.status.value_counts().to_dict(), 'run': run_counts,
    'unmatched_addresses': address_results.loc[address_results.status.ne('matched'), ['address_query', 'status', 'error_message']].to_dict('records')}
print(json.dumps(report, ensure_ascii=False, indent=2))
display(enriched[['geocode_address_query', 'latitude', 'longitude', 'coordinate_level', 'geocode_status']].head())

{
  "provider": "gsi_address_search",
  "query_version": "chome_v1",
  "generated_at": "2026-10-08T08:25:58.463602+00:00",
  "listing_count": 17380,
  "unique_address_count": 87,
  "matched_address_count": 87,
  "matched_listing_count": 17380,
  "invalid_input_listing_count": 0,
  "address_coverage": 1.0,
  "listing_coverage": 1.0,
  "address_status_counts": {
    "matched": 87
  },
  "run": {
    "cache_hits": 93,
    "lookups": 0
  },
  "unmatched_addresses": []
}


,geocode_address_query,latitude,longitude,coordinate_level,geocode_status
0,大阪府大阪市淀川区東三国2丁目,34.743404,135.502136,chome,matched
1,大阪府大阪市淀川区西三国3丁目,34.740242,135.485992,chome,matched
2,大阪府大阪市淀川区西三国3丁目,34.740242,135.485992,chome,matched
3,大阪府大阪市淀川区三津屋中1丁目,34.728222,135.472641,chome,matched
4,大阪府大阪市淀川区加島1丁目,34.725845,135.459808,chome,matched


## Bước 8 — Xuất dữ liệu đã bổ sung
Giữ nguyên CSV current nguồn của notebook 02. Cache và outputs ở `notebook/data` được Git ignore. CSV theo địa chỉ không chứa JSON phản hồi; JSON audit đầy đủ nằm trong cache.

In [9]:
enriched.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
address_results.to_csv(ADDRESS_OUTPUT_PATH, index=False, encoding='utf-8-sig')
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
for path in [OUTPUT_PATH, ADDRESS_OUTPUT_PATH, REPORT_PATH, CACHE_PATH]:
    print(path.resolve())

/Users/hoangvinh/Library/CloudStorage/OneDrive-Personal/Workspace/japan_analysis/notebook/data/suumo_rentals_current_geocoded.csv
/Users/hoangvinh/Library/CloudStorage/OneDrive-Personal/Workspace/japan_analysis/notebook/data/suumo_address_geocodes.csv
/Users/hoangvinh/Library/CloudStorage/OneDrive-Personal/Workspace/japan_analysis/notebook/data/suumo_geocode_report.json
/Users/hoangvinh/Library/CloudStorage/OneDrive-Personal/Workspace/japan_analysis/notebook/data/suumo_address_geocodes_cache.json


## Chuyển sang Airflow/dbt
Pipeline dùng cùng quy tắc `chome_v1` trên thành phần địa lý từ dbt và lưu cache ở `raw.suumo_address_geocodes`. Airflow geocode sau mart observations và trước dim geography. Tọa độ nằm ở `dim_geography`; fact dùng `geography_key`, views xuất sẵn tọa độ.

Xem runbook `docs/suumo-geocoding.md` để bootstrap cache, backfill và chạy lại. Nâng độ chính xác đến tòa nhà cần dữ liệu địa chỉ chi tiết hơn ở một đợt bổ sung riêng.